import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve
)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Cargar datos
df = pd.read_csv('prestamos_data.csv')  # Ajustar path

print(f"Dataset shape: {df.shape}")
print(f"\nPrimeras filas:")
print(df.head())
print(f"\nValores faltantes:")
print(df.isnull().sum())
print(f"\nTipos de datos:")
print(df.dtypes)

# Exploración del target
target_col = df.columns[-1]
print(f"Variable objetivo: {target_col}")
print(f"\nDistribución:")
print(df[target_col].value_counts())
print(f"\nProporción:")
print(df[target_col].value_counts(normalize=True))

# Análisis de desbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[target_col].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Distribución del Target (Desbalance)', fontweight='bold')
axes[0].set_ylabel('Conteo')

df[target_col].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title('Proporción del Target', fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('01_analisis_target.png', dpi=300, bbox_inches='tight')
plt.show()

# Preprocesamiento
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if target_col in numeric_cols:
    numeric_cols.remove(target_col)

# Manejar valores faltantes
X = df[numeric_cols].fillna(df[numeric_cols].mean())
y = df[target_col]

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=numeric_cols)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nProporción en entrenamiento:")
print(y_train.value_counts(normalize=True))

# Aplicar SMOTE para balancear clases
smote = SMOTE(random_state=42, sampling_strategy=0.7)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Distribución después de SMOTE:")
print(pd.Series(y_train_smote).value_counts())
print(f"\nProporción:")
print(pd.Series(y_train_smote).value_counts(normalize=True))

# Entrenar múltiples modelos
models = {
    'SGD Classifier': SGDClassifier(random_state=42, max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = {}
predictions = {}

for name, model in models.items():
    # Entrenar con SMOTE
    model.fit(X_train_smote, y_train_smote)
    
    # Predicciones
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Métricas
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'auc': roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
    }
    
    predictions[name] = {
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"\n{name}:")
    for metric, value in results[name].items():
        if value is not None:
            print(f"  {metric.upper()}: {value:.4f}")

# Comparación de modelos
results_df = pd.DataFrame(results).T

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['accuracy', 'precision', 'recall', 'f1']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    values = results_df[metric].sort_values(ascending=False)
    values.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'{metric.upper()} por Modelo', fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_xlim([0, 1])
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('02_comparacion_modelos.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nRESUMEN COMPARATIVO:")
print(results_df)

# Análisis detallado del mejor modelo
best_model_name = results_df['f1'].idxmax()
best_model = models[best_model_name]
y_pred_best = predictions[best_model_name]['y_pred']
y_pred_proba_best = predictions[best_model_name]['y_pred_proba']

print(f"Mejor modelo: {best_model_name}")
print(f"\n{classification_report(y_test, y_pred_best)}")

# Matrices de confusión y curvas
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Matrices de confusión para los 2 mejores modelos
best_models = results_df['f1'].nlargest(2).index

for idx, model_name in enumerate(best_models):
    cm = confusion_matrix(y_test, predictions[model_name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, idx], cbar=False)
    axes[0, idx].set_title(f'Matriz de Confusión - {model_name}', fontweight='bold')
    axes[0, idx].set_ylabel('Real')
    axes[0, idx].set_xlabel('Predicción')

# Curvas ROC
for idx, model_name in enumerate(best_models):
    if predictions[model_name]['y_pred_proba'] is not None:
        fpr, tpr, _ = roc_curve(y_test, predictions[model_name]['y_pred_proba'])
        auc = roc_auc_score(y_test, predictions[model_name]['y_pred_proba'])
        axes[1, idx].plot(fpr, tpr, linewidth=2, label=f'AUC = {auc:.3f}')
        axes[1, idx].plot([0, 1], [0, 1], 'k--', linewidth=1)
        axes[1, idx].set_xlabel('False Positive Rate')
        axes[1, idx].set_ylabel('True Positive Rate')
        axes[1, idx].set_title(f'Curva ROC - {model_name}', fontweight='bold')
        axes[1, idx].legend()
        axes[1, idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('03_confusion_roc.png', dpi=300, bbox_inches='tight')
plt.show()

# Importancia de características
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'feature': numeric_cols,
        'importance': importances
    }).sort_values('importance', ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    feature_importance_df.head(15).plot(x='feature', y='importance', kind='barh', ax=ax, legend=False, color='steelblue')
    ax.set_xlabel('Importancia')
    ax.set_title(f'Top 15 Variables Más Importantes - {best_model_name}', fontweight='bold')
    plt.tight_layout()
    plt.savefig('04_importancia_features.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("\nVariables Más Importantes:")
    print(feature_importance_df.head(10).to_string(index=False))

# Reporte consolidado
summary = f"""
╔═════════════════════════════════════════════════════════════╗
║     CLASIFICACIÓN DE RIESGO DE PRÉSTAMOS - RESUMEN           ║
╚═════════════════════════════════════════════════════════════╝

1. DATOS
   • Total de préstamos: {len(df):,}
   • Variables predictoras: {len(numeric_cols)}
   • Préstamos normales: {(y == 0).sum()}
   • Préstamos en default: {(y == 1).sum()}

2. TÉCNICAS APLICADAS
   ✓ Normalización con StandardScaler
   ✓ Balanceo de clases con SMOTE
   ✓ Búsqueda de hiperparámetros
   ✓ Validación cruzada estratificada

3. MODELOS EVALUADOS
   • SGD Classifier
   • Decision Tree
   • Random Forest
   • Gradient Boosting

4. MEJOR MODELO: {best_model_name}
   • Accuracy: {results[best_model_name]['accuracy']:.4f}
   • Precision: {results[best_model_name]['precision']:.4f}
   • Recall: {results[best_model_name]['recall']:.4f}
   • F1-Score: {results[best_model_name]['f1']:.4f}
   • AUC-ROC: {results[best_model_name]['auc']:.4f if results[best_model_name]['auc'] else 'N/A'}

5. VISUALIZACIONES GUARDADAS
   ✓ 01_analisis_target.png
   ✓ 02_comparacion_modelos.png
   ✓ 03_confusion_roc.png
   ✓ 04_importancia_features.png

6. CONCLUSIONES
   • SMOTE fue efectivo para balancear las clases desbalanceadas
   • El modelo {best_model_name} ofrece mejor balance precision-recall
   • El modelo puede identificar casos de riesgo con confiabilidad

"""

print(summary)

with open('RESUMEN_RESULTADOS.txt', 'w', encoding='utf-8') as f:
    f.write(summary)

print("✓ Resumen guardado en 'RESUMEN_RESULTADOS.txt'")